In [1]:
import pandas as pd
import gc
import csv
import sys
import os
# CRITICAL FIX: Increase max field size to handle very long articles/content
csv.field_size_limit(sys.maxsize)

input_file = r"H:\\news.csv\\news.csv\\news_cleaned_2018_02_13.csv"
output_file = r"H:\\data_cset312\\news_fake_real_only.csv"

chunksize = 50000          # small → less likely to crash; increase to 100000 if stable
usecols = ['title', 'content', 'type']

print("Input file exists?", os.path.exists(input_file))
print(f"Starting processing...\nInput:  {input_file}\nOutput: {output_file}")
print("Keeping only 'fake' and 'reliable' (renamed to 'real')")
print("Watch Task Manager → Memory usage. Close other apps if possible.")

Input file exists? True
Starting processing...
Input:  H:\\news.csv\\news.csv\\news_cleaned_2018_02_13.csv
Output: H:\\data_cset312\\news_fake_real_only.csv
Keeping only 'fake' and 'reliable' (renamed to 'real')
Watch Task Manager → Memory usage. Close other apps if possible.


In [2]:
import csv

first_chunk = True
total_kept = 0

print("Field size limit increased → starting processing...")

for chunk_num, chunk in enumerate(
    pd.read_csv(
        input_file,
        chunksize=chunksize,
        usecols=usecols,
        low_memory=True,
        on_bad_lines='skip',
        encoding_errors='replace',
        engine='python',
        quoting=csv.QUOTE_ALL,
    ),
    start=1
):
    kept = chunk[chunk['type'].isin(['fake', 'reliable'])].copy()
    
    if kept.empty:
        print(f"Chunk {chunk_num:4d}: skipped (0 matching rows)")
        continue
    
    kept['label'] = kept['type'].replace({'reliable': 'real'})
    kept['text'] = (kept['title'].fillna('') + ' ' + kept['content'].fillna('')).str.strip()
    
    out_chunk = kept[['text', 'label']]
    
    out_chunk.to_csv(
        output_file,
        mode='w' if first_chunk else 'a',
        header=first_chunk,
        index=False,
        encoding='utf-8'
    )
    
    kept_count = len(out_chunk)
    total_kept += kept_count
    print(f"Chunk {chunk_num:4d} | Kept {kept_count:6,d} rows | Total so far: {total_kept:8,d}")
    
    del chunk, kept, out_chunk
    gc.collect()
    
    first_chunk = False

print(f"\nProcessing finished!")
print(f"Total rows kept: {total_kept:,}")
print(f"Output saved to: {output_file}")

Field size limit increased → starting processing...
Chunk    1 | Kept 25,693 rows | Total so far:   25,693
Chunk    2 | Kept 20,364 rows | Total so far:   46,057
Chunk    3 | Kept    302 rows | Total so far:   46,359
Chunk    4 | Kept    688 rows | Total so far:   47,047
Chunk    5 | Kept    207 rows | Total so far:   47,254
Chunk    6 | Kept  6,139 rows | Total so far:   53,393
Chunk    7 | Kept     92 rows | Total so far:   53,485
Chunk    8 | Kept 15,260 rows | Total so far:   68,745
Chunk    9 | Kept  3,618 rows | Total so far:   72,363
Chunk   10 | Kept    173 rows | Total so far:   72,536
Chunk   11 | Kept  7,143 rows | Total so far:   79,679
Chunk   12 | Kept    266 rows | Total so far:   79,945
Chunk   13 | Kept  1,679 rows | Total so far:   81,624
Chunk   14 | Kept 34,285 rows | Total so far:  115,909
Chunk   15 | Kept  6,220 rows | Total so far:  122,129
Chunk   16 | Kept  3,031 rows | Total so far:  125,160
Chunk   17 | Kept    144 rows | Total so far:  125,304
Chunk   18 | 

In [3]:
# Cell: Quick look at processed file
import pandas as pd

# Head + basic info
df = pd.read_csv(output_file, nrows=5)
print("First 5 rows:")
print(df.to_string(index=False))

print("\nColumns:", df.columns.tolist())

First 5 rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5 non-null      str  
 1   label   5 non-null      str  
dtypes: str(2)
memory usage: 16.1 KB


In [5]:
# Cell: Label distribution
labels = pd.read_csv(output_file, usecols=['label'])['label']

print("Label value counts:")
print(labels.value_counts(dropna=False))

print("\nPercentages:")
print(labels.value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

print(f"\nTotal rows in file: {len(labels):,}")

Label value counts:
label
real    1913222
fake     894746
Name: count, dtype: int64

Percentages:
label
real    68.14 %
fake    31.86 %
Name: proportion, dtype: str

Total rows in file: 2,807,968


In [6]:
texts = pd.read_csv(output_file, usecols=['text'])
texts['length'] = texts['text'].astype(str).str.len()

print("Text length stats:")
print(texts['length'].describe().round(0))

print("\nLongest text preview (first 300 chars):")
print(texts.loc[texts['length'].idxmax(), 'text'][:300] + "...")

Text length stats:
count    2807955.0
mean        2941.0
std         3557.0
min           47.0
25%          834.0
50%         1941.0
75%         4083.0
max       196884.0
Name: length, dtype: float64

Longest text preview (first 300 chars):
Take It From a Liberal Redneck Be sure to check out this video—a post-election, Thanksgiving message that’s right in line with what readers have been debating in this Notes thread (except now with delightful lines like “we can keep shoving our heads up our high-horses asses all we want”). The self-d...
